# ساهر — توليد الصور من Kaggle

هذا الدفتر يحول المشروع إلى توليد **صور ثابتة** من وصف نصي باستخدام ComfyUI وStable Diffusion 1.5 FP16. يتعامل المستخدم عبر خلايا الدفتر: يعدّل `PROMPT`، ينتظر، ثم يشاهد الصورة ورابط تنزيلها ضمن المخرجات. **لا توجد صفحة ويب مستقلة، ولا Tunnel، ولا واجهة عامة.**

**خطوات البداية:** من إعدادات Notebook اختر Accelerator → GPU وفعّل Internet؛ شغّل الخلايا بالترتيب، ثم عدّل `PROMPT` قبل خلية التوليد. لا تتوفر موارد Kaggle في كل وقت وقد تختلف بطاقة GPU. النموذج الحالي يُنتج 512×512 عند 16 خطوة، وهو أخف من تجربة الفيديو لكنه قديم مقارنة بالنماذج الأحدث.

**القيود:** لا يضيف تطبيق Saher قائمة حظر كلمات خاصة به، ويمرر prompt إلى ComfyUI؛ لكن لا يمكن ضمان إنتاج كل محتوى. رخصة Stable Diffusion 1.5 هي CreativeML OpenRAIL-M وبها شروط استخدام، كما تسري سياسات Kaggle والقوانين. لا تستخدم صور أشخاص أو محتوى مؤذٍ دون إذن.

## 1. فحص وجود GPU والذاكرة قبل تنزيل النموذج


In [ ]:
import sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('لم يتوفر GPU في هذه الجلسة. اختر Runtime > Change runtime type > GPU، أو حاول لاحقاً؛ التوفر المجاني غير مضمون.')
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
print('GPU:', props.name, '| VRAM GiB:', round(vram_gib, 1))
if vram_gib < 6:
    raise RuntimeError(f'الذاكرة المعروضة {vram_gib:.1f} GiB أقل من الحد التقريبي الموثق لمسار SD1.5 في ComfyUI (نحو 6 GiB). أوقف التجربة لتجنب تنزيل الأوزان ثم الفشل.')
print('GPU check passed. This is only a hardware check, not a generation benchmark.')

## 2. تثبيت ComfyUI وSaher Image

تحقق من تفعيل خيار Internet في إعدادات Kaggle قبل تشغيل الخلية؛ نحتاجه لتنزيل الشيفرة والـcheckpoint.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git
!git clone --depth 1 --branch feature/saher-mvp-foundation https://github.com/jonbander/saher-video-colab.git
%pip install -r /kaggle/working/ComfyUI/requirements.txt
%pip install huggingface_hub
%pip install -e /kaggle/working/saher-video-colab

## 3. تنزيل checkpoint الصور

يستخدم هذا المسار ملف Stable Diffusion 1.5 FP16 الرسمي الذي توصي به وثائق ComfyUI للتجربة الأساسية. حجمه عدة غيغابايت؛ ينزّل إلى مساحة Kaggle المؤقتة.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

COMFY = Path('/kaggle/working/ComfyUI')
MODEL_REPO = 'Comfy-Org/stable-diffusion-1.5'
MODEL_FILE = 'v1-5-pruned-emaonly-fp16.safetensors'
model_dir = COMFY / 'models' / 'checkpoints'
model_dir.mkdir(parents=True, exist_ok=True)
revision = HfApi().model_info(MODEL_REPO).sha
print('Pinned checkpoint repository revision:', revision)
model_path = model_dir / MODEL_FILE
if not model_path.exists():
    model_path = Path(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, revision=revision, local_dir=str(model_dir)))
print(f'Model ready: {model_path} ({model_path.stat().st_size/1024**3:.2f} GiB)')

## 4. تحديد workflow وقيم تجربة سريعة

ملف workflow مضمن في المستودع. يولد صورة واحدة بدقة 512×512 وبـ16 خطوة، ويكتب الصورة إلى مجلد إخراج ComfyUI.

In [ ]:
from pathlib import Path
REPO = Path('/kaggle/working/saher-video-colab')
workflow_path = REPO / 'workflows' / 'sd15-image-api.json'
assert workflow_path.is_file(), f'لم يوجد workflow: {workflow_path}'
print('Saher image workflow:', workflow_path)

## 5. تشغيل ComfyUI على loopback داخل جلسة Kaggle


In [ ]:
import subprocess, time, requests, sys

log_file = open('/kaggle/working/comfyui.log', 'w')
comfy_process = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT)
base = 'http://127.0.0.1:8188'
for attempt in range(90):
    if comfy_process.poll() is not None:
        raise RuntimeError('ComfyUI توقف مبكراً. آخر سجل:\n' + Path('/kaggle/working/comfyui.log').read_text(errors='replace')[-5000:])
    try:
        health = requests.get(base + '/system_stats', timeout=3)
        if health.ok:
            print('ComfyUI API جاهز محلياً؛ لم يتم كشف منفذ عام.')
            print('PyTorch:', health.json().get('system',{}).get('pytorch_version','غير معروف'))
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError('انتهت مهلة بدء ComfyUI. افحص /kaggle/working/comfyui.log.')

## 6. توليد صورة وعرضها هنا

عدّل الوصف التالي، ثم شغّل هذه الخلية. النص يرسل كما هو بعد إزالة الفراغات الطرفية فقط؛ فحص API يفرض طولاً من 3 إلى 4000 محرف ولا يضيف قائمة حظر محتوى.

In [ ]:
import os, time
from fastapi.testclient import TestClient

PROMPT = 'A small cabin in a quiet forest at sunset, detailed digital illustration, warm light'  # عدّل هذا الوصف
if len(PROMPT.strip()) < 3 or len(PROMPT.strip()) > 4000:
    raise ValueError('يجب أن يكون الوصف بين 3 و4000 محرف.')
os.environ['COMFYUI_BASE_URL'] = 'http://127.0.0.1:8188'
os.environ['COMFYUI_WORKFLOW_PATH'] = str(workflow_path)
os.environ['COMFYUI_PROMPT_NODE_ID'] = '6'
os.environ['COMFYUI_PROMPT_INPUT'] = 'text'

import importlib, saher_image.app
importlib.reload(saher_image.app)
client = TestClient(saher_image.app.app)
prompt = PROMPT.strip()
submitted = client.post('/api/v1/jobs', json={'prompt': prompt})
assert submitted.status_code == 202, submitted.text
job_id = submitted.json()['job_id']
print('Saher job:', job_id)

job = None
for attempt in range(120):  # حتى 10 دقائق؛ قد تنتهي جلسة Kaggle قبل ذلك.
    response = client.get(f'/api/v1/jobs/{job_id}')
    if response.status_code != 200:
        raise RuntimeError(response.text)
    job = response.json()
    if attempt % 3 == 0:
        print(f"Elapsed ~{attempt*5}s | state={job['state']}")
    if job['state'] == 'completed':
        break
    if job['state'] == 'failed':
        raise RuntimeError(job.get('error') or 'فشل التوليد داخل ComfyUI.')
    if job['state'] == 'unknown':
        raise RuntimeError('المهمة غير موجودة في طابور ComfyUI أو سجله.')
    time.sleep(5)
else:
    raise TimeoutError('انتهت مهلة الانتظار؛ افحص /kaggle/working/comfyui.log.')

images = [item for item in job['outputs'] if item['content_type'].startswith('image/')]
if not images:
    raise RuntimeError(f'لا توجد صورة مخرجة. تحقق من عقدة SaveImage. Outputs: {job["outputs"]}')
asset = client.get(images[0]['url'])
assert asset.ok, asset.text[:1000]
output_path = Path('/kaggle/working/saher-image.png')
output_path.write_bytes(asset.content)
print(f'تم حفظ الصورة: {output_path} ({output_path.stat().st_size/1024**2:.2f} MiB)')
from IPython.display import FileLink, Image, display
display(Image(filename=str(output_path)))
display(FileLink(str(output_path)))

## 7. إيقاف ComfyUI

بعد تنزيل الصورة، شغّل هذه الخلية لإنهاء خادم ComfyUI المحلي.

In [ ]:
if 'comfy_process' in globals() and comfy_process.poll() is None:
    comfy_process.terminate()
    try:
        comfy_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        comfy_process.kill()
    print('ComfyUI stopped.')
else:
    print('ComfyUI is not running.')

### المصادر

- [دليل ComfyUI الرسمي لتحويل النص إلى صورة وSD1.5](https://docs.comfy.org/tutorials/basic/text-to-image)
- [بطاقة Stable Diffusion 1.5 المعاد تغليفها لـComfyUI](https://huggingface.co/Comfy-Org/stable-diffusion-1.5)
- [توثيق Kaggle Notebooks](https://www.kaggle.com/docs/notebooks)
- [Saher — فرع الصور](https://github.com/jonbander/saher-video-colab/tree/feature/saher-mvp-foundation)